# CodeLlama-7B QLoRA Fine-Tuning
Paper: *Optimizing Python Code Completion with Parameter-Efficient Fine-Tuning*

Run cells top-to-bottom. Training checkpoints are saved to Google Drive automatically.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Step 2: Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/sedanurkilic/Python-Code-Completion-CodeLlama-7B-.git'
REPO_DIR = '/content/Python-Code-Completion-CodeLlama-7B-'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
print('Working directory:', os.getcwd())

## Step 3: Install Dependencies

 Runtime will restart after installation. Re-run from Step 4 after restart.

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed. If prompted, restart the runtime and re-run from Step 4.')

## Step 4: Set Up Drive Persistence

Symlinks `./outputs/` to Google Drive so checkpoints survive Colab disconnections.

In [ ]:
import os

REPO_DIR = '/content/Python-Code-Completion-CodeLlama-7B-'
%cd $REPO_DIR

DRIVE_OUTPUTS = '/content/drive/MyDrive/codellama-qlora/outputs'
LOCAL_OUTPUTS = os.path.join(REPO_DIR, 'outputs')

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

if os.path.islink(LOCAL_OUTPUTS):
    print('Symlink already exists, skipping.')
elif os.path.isdir(LOCAL_OUTPUTS):
    print('outputs/ directory exists locally (not symlinked). Checkpoints will NOT persist to Drive.')
else:
    os.symlink(DRIVE_OUTPUTS, LOCAL_OUTPUTS)
    print(f'Symlinked: {LOCAL_OUTPUTS} -> {DRIVE_OUTPUTS}')

## Step 5: Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')

## Step 6: Verify Config

In [ ]:
from src.config import load_config
cfg = load_config()
print(f'Model: {cfg.training.model_name}')
print(f'LoRA r={cfg.lora.r}, alpha={cfg.lora.lora_alpha}, targets={cfg.lora.target_modules}')
print(f'Epochs: {cfg.training.num_train_epochs}, LR: {cfg.training.learning_rate}')
print(f'4-bit NF4: {cfg.quantization.load_in_4bit}')

## Step 7: Train

⏱️ Estimated time: 6-10 hours on T4. Checkpoints saved every 100 steps to Drive.

In [ ]:
!python src/train.py

## Step 8: Evaluate on HumanEval

Generates 10 samples per problem and computes pass@1, pass@5, pass@10.

In [ ]:
!python scripts/run_evaluation.py --adapter_path ./outputs/final --output results/samples.jsonl

## Step 9: Display Results

In [ ]:
import json

with open('results/metrics.json') as f:
    metrics = json.load(f)

print('\n=== HumanEval Results ===')
print(f"pass@1:  {metrics.get('pass@1', 'N/A'):.4f}")
print(f"pass@5:  {metrics.get('pass@5', 'N/A'):.4f}")
print(f"pass@10: {metrics.get('pass@10', 'N/A'):.4f}")